# Tic tac toe
### Steven Fowler <a id='headerlink' href="https://stevenafowler.github.io/profile/" target="_blank">(see more projects here)</a>

<img id = 'headerimage' src="assets/images/HeaderTicTacToe_ex.svg">

Tic tac toe is a classic m,n,k-game that provides a great foundation for AI based models to become players. The following shows an implementation of the game with 3 player-models that can compete against each other:
* A random model that is based all on luck,
* A minimax model that I cannot beat,
* And a neural network the matches the minimax model in performance.

### What I did
* Created a game engine of tic tac toe with a player that selects moves at random to test the game.
* Wrote a minimax-player to make moves based on highest score calculated by recursive exploration of the current game state.
* I decided to convert the players to a class structure based on a abstract class, this making running the game with different players easy to modify.
* Created a neural network model based on the minimax-player's recursive decision making.
* Look at the performance of all the player-models.

### Why
* For me, Reinforcement Learning is one the more impressive examples of machine learning especially when I consider a  model like DeepMind's AlphaZero that beat the world champion engine (Stockfish) with < 9 hours of training.
* So, this is my exploration of a much simpler game!

### What I learnt
* The early choice of how I represented the board-state (as a simple 1d array): 
    - Proved to make analysis of the game state very simple, 
    - The recursive minimax model could be coded in fewer lines (1 model for both players)
    - Integration into the neural network was simple as the 1d array can feed directly into the model
    - So, it was a reminder that early planning and consideration of architecture can have an significant impact on implementation.
* A neural network trained against a minimax algorithm can perform with equivalence to the minimax model.

### Further work
Implement/add:
* Create a model-player based on Q-learning.

## Defining the game engine

The following defines the game board as a 1D array. There are also a few generic display functions to display the game board.

In [3]:
import numpy as np

"""
Definition of the board (state) of the game.
2 display options

"""

# Game board
board = np.zeros(9, dtype=np.int8)

def display_board(board):
    # Simple solution
    print(f"{board[0]} | {board[1]} | {board[2]}")
    print(f"{board[3]} | {board[4]} | {board[5]}")
    print(f"{board[6]} | {board[7]} | {board[8]}")
    print()

def display_board_characters(board):
    strBoard = np.empty(9, dtype='U1')
    strBoard[np.where(board == 1)[0]] = 'X'
    strBoard[np.where(board == 0)[0]] = ' '
    strBoard[np.where(board == -1)[0]] = 'O'

    # Simple solution
    display_board(board=strBoard)


# Test board display
board[0] = 1
board[1] = 1
board[2] = 1
board[8] = -1

print("Testing board display functions...\n")
print(f"Game board {board}\n")

display_board(board=board)
display_board_characters(board=board)

Testing board display functions...

Game board [ 1  1  1  0  0  0  0  0 -1]

1 | 1 | 1
0 | 0 | 0
0 | 0 | -1

X | X | X
  |   |  
  |   | O



The function for check the state of the board is below. This returns a value that is used by the minimax model to score the recursive learning of possible plays.

The assessment of the state is achieved by the matrix multiplication of the board array against a solution-matrix. The solution-matrix represents the 8 possible ways to wins the game (3x rows, 3x columns & 2x diagonals). For example, a value of 3 or -3 represents the X or O respectively winning the game.

In [ ]:

"""
Function that checks the board (state) to determine if:

    x There is a winner
    x The game is a draw
    x The game has not finished

If the game is finished a score is allocated - this is the score used:

    x In the minimax model for recursive learning
    x In the game to ID which player won

"""

def check_game_complete(board, solutionMatrix):

    depth = np.sum(np.abs(board))   # No. of turns taken

    # Look at results
    scores = solutionMatrix.dot(board)

    if np.max(scores) == 3:     # X has won
        return 10 - depth
    elif np.min(scores) == -3:  # O has won
        return -10 + depth
    elif depth == 9:            # Draw
        return 0

    return None     # Game not finished

###########################################################
# Create solution matrix

def encode_array(arrIndices):
    # Function encodes array of indices into array of ones
    #
    #   arrIndices: Indices array
    #
    rtn = np.zeros(9)
    rtn[arrIndices] = 1
    return rtn

# Define solutions as indices of the board    
solutions = [[0, 1, 2], [3, 4, 5], [6, 7, 8],   # Rows
             [0, 3, 6], [1, 4, 7], [2, 5, 8],   # Columns
             [0, 4, 8], [2, 4, 6]]              # Diagonals

# Encode indices into arrays of 1s & 0s to form solutionMatrix
solMat = np.apply_along_axis(encode_array, axis=1, arr=solutions, )

###########################################################


# Test board (state) evaluation with solutionMatrix
print("Testing board check - Simulate X wins\n")
testBoardXWin = np.array([1, 1, 1, 0, -1, 0, -1, 0, 0])
display_board_characters(testBoardXWin)
print(f"{check_game_complete(board=testBoardXWin, solutionMatrix=solMat)} <- Positive value so X wins\n")

print("Testing board check - Simulate game not finished\n")
testBoardNone = np.array([1, -1, 1, 0, -1, 0, -1, 0, -1])
display_board_characters(testBoardNone)
print(f"{check_game_complete(board=testBoardNone, solutionMatrix=solMat)} <- None means game not concluded\n")

print("Testing board check - Simulate O wins\n")
testBoardOWin = np.array([-1, 1, 1, 0, -1, 0, -1, 0, -1])
display_board_characters(testBoardOWin)
print(f"{check_game_complete(board=testBoardOWin, solutionMatrix=solMat)} <- Negative value so O wins\n")

print("Testing board check - Simulate a draw\n")
testBoardDraw = np.array([1, -1, 1, -1, -1, 1, -1, 1, -1])
display_board_characters(testBoardDraw)
print(f"{check_game_complete(board=testBoardDraw, solutionMatrix=solMat)} <- Zero value so a draw\n")

Testing board check - Simulate X wins

X | X | X
  | O |  
O |   |  

5 <- Positive value so X wins

Testing board check - Simulate game not finished

X | O | X
  | O |  
O |   | O

None <- None means game not concluded

Testing board check - Simulate O wins

O | X | X
  | O |  
O |   | O

-4 <- Negative value so O wins

Testing board check - Simulate a draw

X | O | X
O | O | X
O | X | O

0 <- Zero value so a draw



### Random player

The random-player is defined below: In summary, the model selects at random the next move from a list of possible moves available. There is no logic of intent of winning :)

In [5]:
def random_player(board, XorO):
    """
    Player function - Takes in the current board and return a move.

    random_player: Move is randomly selected

    Args:
        board (array(9, int)): Array of int that represent the current board
        XorO (int): value of 1 or -1, corresponding to X and O respectively

    Returns:
        int: index of move with respect to board
    """
    possibleMoves = np.where(board == 0)[0]
    move = np.random.choice(possibleMoves)
    return move

The game loop is defined below.

In [6]:
# Game loop

def playGame(playerX, playerO, solMat, startBoard, startPlayer = 1):

    plyBoard = startBoard
    player = startPlayer

    # Loop taking turns
    for t in range(9):   # Max of 9 turns, while loop would work but this helps debugging
        # Player makes move
        if player == 1:
            plyBoard[playerX(plyBoard, player)] = player
        else: # player == -1
            plyBoard[playerO(plyBoard, player)] = player
        
        # Evaluate game (has anyone won?)
        result = check_game_complete(board=plyBoard, solutionMatrix=solMat)

        if result is None:      # Evaluate 'None' first as cannot be used in equality
            player = -player    # Change player
            continue            # Skip to next turn          
        elif result > 0:
            print("Game is won by X (1)")
            display_board_characters(plyBoard)
            break               
        elif result < 0:
            print("Game is won by O (-1)")
            display_board_characters(plyBoard)
            break
        elif result == 0:
            print("Game is a draw")
            display_board_characters(plyBoard)
            break
        else:
            raise ValueError("Evaluation failed to conclude!")   # Should create custom exception :)
        
    return None # Will be updated when I need game data for Neural Network

This is an example of 2 random-players competing against each other. Although the move is random it can be seen that player 1, who goes first, tends to win more games than player 2. This illustrates that the game is biased towards the player that has the first turn.

In [7]:
# Random vs Random players

noGames = 5
# Loop number of games
for n in range(noGames):

    rndBoard = np.zeros(9, dtype=int)   # Set board 

    playGame(playerX=random_player, playerO=random_player, solMat=solMat, startBoard=rndBoard)

Game is won by X (1)
O | X |  
  | X | O
X | X | O

Game is won by X (1)
O |   | O
O | X |  
X | X | X

Game is won by X (1)
  | O | X
  | X | X
O | O | X

Game is won by O (-1)
O | X |  
  | O |  
X | X | O

Game is won by X (1)
X | X | X
  | O |  
O | O | X



## Move to class structure

Due to the advantage of having a class structure for the minimax and neural network an abstract class was created for a player. This allows different player to be easily pitted against each other. To use this class-player structure the random-player and the game loop are also re-written below.

In [8]:
from abc import ABC, abstractmethod

# Blueprint for tic tac toe player
class player(ABC):
    @abstractmethod
    def nextMove(self, board):
        """
        Calculated next move based on current state (board)
        Args:
            board (array(9, int)): Array of int that represent the current board
        
        Returns:
            int: index of move with respect to board
            
        """
        pass

In [9]:
# Random player
class RandomPlayer(player):
    
    def __init__(self):
        pass

    def nextMove(self, board):
        possibleMoves = np.where(board == 0)[0]
        move = np.random.choice(possibleMoves)
        return move

In [10]:
def playGameClass(playerX, playerO, solMat, startBoard, startPlayer = 1):

    plyBoard = startBoard
    player = startPlayer
    result = None

    # Loop taking turns
    for t in range(9):   # Max of 9 turns, while loop would work but this helps debugging
        # Player makes move
        if player == 1:
            plyBoard[playerX.nextMove(plyBoard)] = player
        else: # player == -1
            plyBoard[playerO.nextMove(plyBoard)] = player
        
        # Evaluate game (has anyone won?)
        result = check_game_complete(board=plyBoard, solutionMatrix=solMat)

        if result is None:      # Evaluate 'None' first as cannot be used in equality
            player = -player    # Change player
            continue            # Skip to next turn          
        elif result > 0:
            print(f"Game is won by X (1), first move was {startPlayer}")
            display_board_characters(plyBoard)
            break               
        elif result < 0:
            print(f"Game is won by O (-1), first move was {startPlayer}")
            display_board_characters(plyBoard)
            break
        elif result == 0:
            print(f"Game is a draw, first move was {startPlayer}")
            display_board_characters(plyBoard)
            break
        else:
            raise ValueError("Evaluation failed to conclude!")   # Should create custom exception :)
        
    return result # Will be updated when I need game data for Neural Network

Testing random-player using the new class structure.

In [11]:
player1X = RandomPlayer()
player2O = RandomPlayer()

for n in range(noGames):

    rndBoard = np.zeros(9, dtype=int)   # Set board 

    playGameClass(playerX=player1X, playerO=player2O, solMat=solMat, startBoard=rndBoard)

Game is won by X (1), first move was 1
O | X | X
O | X | O
X | O | X

Game is won by X (1), first move was 1
  | O |  
X | X | X
O | O | X

Game is won by X (1), first move was 1
O | X |  
O | O |  
X | X | X

Game is won by O (-1), first move was 1
O | O | O
X |   | X
X | O | X

Game is won by X (1), first move was 1
X | X | X
X | O | O
O | X | O



### minimax

Minimax is a model used to minimize loss for worst case scenarios. In the context of Tic Tac Toe, all next possible move for a player are scored based on a recursive assessment of possible future outcomes.

The following is an implementation of the minimax algorithm for a tic tac toe game.

In [ ]:
# MaxMin player
class MiniMaxPlayer(player):
    
    def __init__(self, playerType, solutionMatrix):
        self.playerType = playerType
        self.solutionMatrix = solutionMatrix
    
    def _minimax(self, board, currentPlayer):
        
        # Evaluate game (has anyone won?)
        result = check_game_complete(board=board, solutionMatrix=self.solutionMatrix)
    
        if result is None:
            
            # Game not concluded, recursively try next move with _minimax
            bestScore = -np.inf #* currentPlayer 
            possMoves = np.where(board == 0)[0]

            for move in possMoves:

                testBoard = board.copy()
                # Try a move
                testBoard[move] = currentPlayer

                # Recursive call of _minimax
                # Note: 'currentPlayer *' ensure bestScore is always a positive value
                bestScore = max(bestScore, 
                                (currentPlayer * self._minimax(board = testBoard, 
                                                              currentPlayer = -currentPlayer)))

            return bestScore * currentPlayer    
        elif result > 0:
            return result             
        elif result < 0:
            return result
        elif result == 0:
            return result
        else:
            raise ValueError("Evaluation failed to conclude!")   # Should create custom exception :)

    def nextMove(self, board):
        
        possibleMoves = np.where(board == 0)[0]
        bestValue = -np.inf
        bestMove = possibleMoves[0]
        scoreTracker = np.ones(9) - 100

        for move in possibleMoves:
            testBoard = board.copy()
            
            # Try a move
            testBoard[move] = self.playerType

            # Score move
            # Note: Current player is updated with '-'
            moveValue = self.playerType * self._minimax(board = testBoard, currentPlayer = -self.playerType)

            scoreTracker[move] = moveValue

            # Update best move
            if moveValue > bestValue:
                bestMove = move
                bestValue = moveValue

        # Add some randomness to player were possible
        # If a draw is the highest score, the draw value selected is random
        if bestValue == 0:
            possMoves = np.where(scoreTracker == 0)[0]
            bestMove = np.random.choice(possMoves)
            
        # print(scoreTracker)

        return bestMove

Test the minimax algorithm: The following state presents a situation where the only way for player O to draw or win the game is to select the middle space.

In [ ]:
# Testing minimax player implementation
 
pValue = -1

player2X = MiniMaxPlayer(playerType = pValue, solutionMatrix=solMat)

testBoard = np.array([1, -1, 1, 0, 0, 0, 0, 0, 0], dtype=np.int8)

move = player2X.nextMove(testBoard)

print("Board state being tested:")
display_board_characters(testBoard)
print(f"Move selected by minimax algorithm: {move}\n")
testBoard[move] = pValue
print("Updated board with O-players move")
display_board_characters(testBoard)

Board state being tested:
X | O | X
  |   |  
  |   |  

Move selected by minimax algorithm: 4

Updated board with O-players move
X | O | X
  | O |  
  |   |  



### Minimax vs Minimax

To test the minimax-player implementation, 2 minimax players are pitted against each other. As can be seen in the below, the minimax vs minimax results in consistent draws.

In [14]:
# minimax vs minimax

player1X = MiniMaxPlayer(playerType = 1, solutionMatrix=solMat)
# player1X = RandomPlayer()
player2O = MiniMaxPlayer(playerType = -1, solutionMatrix=solMat)
# player2O = RandomPlayer()

for n in range(noGames):

    rndBoard = np.zeros(9, dtype=int)   # Set board 

    playGameClass(playerX=player1X, playerO=player2O, solMat=solMat, startBoard=rndBoard, startPlayer=1)

Game is a draw, first move was 1
O | X | X
X | O | O
X | O | X

Game is a draw, first move was 1
O | X | O
X | X | O
X | O | X

Game is a draw, first move was 1
X | O | X
O | O | X
X | X | O

Game is a draw, first move was 1
O | X | O
O | X | X
X | O | X

Game is a draw, first move was 1
X | X | O
O | O | X
X | X | O



## Neural network

The following shows the creation of the neural network model to play tic tac toe. The key aspect to this model is generating data to train it, this is achieved by using the minimax algorithm discussed above. So, it could be argued that the neural network is actually training to simulate the minimax model.

To generate the training data a specific implementation of the minimax is coded below. This implementation returns an array of scores for all possible moves for a given board state.

In [17]:
# minimax a board for training data generation

def dataGenminimax(board, currentPlayer, solutionMatrix):
        
    # Evaluate game (has anyone won?)
    result = check_game_complete(board=board, solutionMatrix=solutionMatrix)

    if result is None:
        
        # Game not concluded, recursively try next move with _minimax
        bestScore = -np.inf #* currentPlayer 
        possMoves = np.where(board == 0)[0]

        for move in possMoves:

            testBoard = board.copy()
            # Try a move
            testBoard[move] = currentPlayer

            # Recursive call of _minimax
            # Note: 'currentPlayer *' ensure bestScore is always a positive value
            bestScore = max(bestScore, 
                            (currentPlayer * dataGenminimax(board = testBoard, 
                                                            currentPlayer = -currentPlayer,
                                                            solutionMatrix=solutionMatrix)))

        return bestScore * currentPlayer    # minimax score
    return result   # End game score (win, draw, lose)

def dataGenNextMoveScore(board, solutionMatrix, player):
        
        currentPlayer = player
        possibleMoves = np.where(board == 0)[0]

        scoreTracker = np.zeros(9) - 20 # move not available = score -20

        for move in possibleMoves:
            testBoard = board.copy()
            
            # Try a move
            testBoard[move] = currentPlayer

            # Score move
            moveValue = currentPlayer * dataGenminimax(board = testBoard, 
                                                      currentPlayer = -currentPlayer, 
                                                      solutionMatrix=solutionMatrix)
            # Update tracker
            scoreTracker[move] = moveValue

        return scoreTracker


# Test data generator

testBoard = np.array([0, -1, 0, 0, 0, 0, 0, 1, 0], dtype=np.int8)

testResult = dataGenNextMoveScore(board=testBoard, solutionMatrix=solMat, player=1)  

print("Board state inputted into training data generator:")
display_board_characters(testBoard)
print(f"Resulting scores: {testResult}")

Board state inputted into training data generator:
  | O |  
  |   |  
  | X |  

Resulting scores: [  0. -20.   0.   0.   0.   0.   0. -20.   0.]


The following is the data generation function. It generates board-states (X input for neural network) by simulating games between a random-player and a minimax-player. Each board-state is evaluated by the minimax function and the scores are stored (true-Y for neural network).

Note: Due to the long time taken to generate data (> 90 minutes), the training data is stored to file for ease of use and is available on GitHub.

In [ ]:
# Data generation

def dataGeneration(playerX, playerO, solutionMatrix, startBoard, startPlayer = 1):

    # dataX = np.empty([9, 9], np.int8)
    # dataY = np.empty([9, 9], np.int8)

    dataX = []
    dataY = []

    # Start a game from a board
    # For each move: record the move (X) & record the minimax score (Y) for training data

    plyBoard = startBoard.copy()
    player = startPlayer

    # Loop through the maximum of 9 moves
    for t in range(9):
        # update board with next move
        if player == 1:
            plyBoard[playerX.nextMove(plyBoard)] = player
        else: # player == -1
            plyBoard[playerO.nextMove(plyBoard)] = player

        player = -player

        dataX.append(plyBoard.tolist())
        dataY.append(dataGenNextMoveScore(board=plyBoard, 
                                          solutionMatrix=solutionMatrix, 
                                          player=player).tolist())
        
        # Evaluate game (has anyone won?)
        result = check_game_complete(board=plyBoard, solutionMatrix=solMat)

        if result is not None:
            return (dataX, dataY)
    

# Test data generation

trainDataX = []
trainDataY = []
tempX = []
tempY = []

player1X = RandomPlayer()
player2O = MiniMaxPlayer(playerType = -1, solutionMatrix=solMat)
# player2O = RandomPlayer()

noGames = 5000

for n in range(noGames):
    startBoard = np.zeros(9, dtype=int)
    tempX, tempY = dataGeneration(player1X, player2O, solMat, startBoard)
    trainDataX.extend(tempX)
    trainDataY.extend(tempY)
    
    if n % 100 == 0:
        print(f"At game {n}")

trainDataX = np.array(trainDataX)
trainDataY = np.array(trainDataY)

print(f"Training data X: {trainDataX.shape}")
print(f"Training data Y: {trainDataY.shape}")

# Would be more efficient to write data directly to file :)
fnTrainDataX = 'trainDataX'
fnTrainDataY = 'trainDataY'
np.save(fnTrainDataX, trainDataX)
np.save(fnTrainDataY, trainDataY)


Training data X: (34967, 9)
Training data Y: (34967, 9)


The neural network player is defined below. As can be seen this player generates its next move using the predict function of a tensorflow model.

In [33]:
# NN player

class nn_v1_player(player):

    def __init__(self, model):
        self.model = model

    def nextMove(self, board):

        # Get prediction from model for current board
        prediction = self.model.predict(board.reshape(1,9), verbose=0)

        # Convert prediction into move and return
        move = np.argmax(prediction)    # return the FIRST best move

        return move

TensorFlow is used to create and train the neural network model based on the generated minimax training data. The model results in a > 99.8% accuracy.

In [32]:
import tensorflow as tf

# Import data from file if needed
trainDataX = np.load(r"tranDataX.npy")
trainDataY = np.load(r"trainDataY.npy")

# trainDataX_unique = np.unique(trainDataX, axis=0)
# trainDataY_unique = np.unique(trainDataY, axis=0)

# one-hot y data
maxIndices = np.argmax(trainDataY, axis=1)
trainDataY_oneHot = tf.one_hot(maxIndices, 9)

#  Build model
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(81, activation='relu'),
    tf.keras.layers.Dense(45, activation='relu'),
    tf.keras.layers.Dense(9)
])

# predictions = model(trainDataX[:1]).numpy()

# tf. nn.softmax(predictions).numpy()

loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

# loss_fn(trainDataY[:1], predictions).numpy()

model.compile(optimizer='adam',
              loss=loss_fn,
              metrics=['accuracy'])

# Train model
model.fit(trainDataX, trainDataY_oneHot, epochs=10)
# model.fit(trainDataX, trainDataY, epochs=10)



Epoch 1/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 707us/step - accuracy: 0.6184 - loss: 1.1872
Epoch 2/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 714us/step - accuracy: 0.9038 - loss: 0.3744
Epoch 3/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 682us/step - accuracy: 0.9646 - loss: 0.1667
Epoch 4/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 708us/step - accuracy: 0.9828 - loss: 0.0936
Epoch 5/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 676us/step - accuracy: 0.9890 - loss: 0.0604
Epoch 6/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 692us/step - accuracy: 0.9939 - loss: 0.0397
Epoch 7/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 690us/step - accuracy: 0.9963 - loss: 0.0272
Epoch 8/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 683us/step - accuracy: 0.9976 - loss: 0.0193
Epoch 9/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 671us/step - accuracy: 0.9981 - loss: 0.0150
Epoch 10/10
1093/1093 ━━━━━━━━━━━━━━━━━━━━ 1s 714us/step - accuracy: 0.9986 - loss: 0.0114


The neural network model is evaluated by simulating games of the random-player vs the neural network player. The random-player always goes first to give it the advantage.

As can be seen below, the random-player never wins a game, most games are won by the NN-player and there are a few draws.

In [34]:
# Evaluate model

# player1X = MiniMaxPlayer(playerType = 1, solutionMatrix=solMat)
player1X = RandomPlayer()
player2O = nn_v1_player(model=model)

countResult = [3]

tempCount = np.zeros(3, dtype=np.int16)

for n in range(50):

    rndBoard = np.zeros(9, dtype=int)   # Set board 

    result = playGameClass(playerX=player1X, playerO=player2O, solMat=solMat, startBoard=rndBoard, startPlayer=1)

    if result > 0:
        tempCount[0] += 1
        # print("Game is won by X (1)")
        # display_board_characters(plyBoard)               
    elif result < 0:
        tempCount[1] += 1
        # print("Game is won by O (-1)")
        # display_board_characters(plyBoard)
    else:
        tempCount[2] += 1
        # print("Game is a draw")
        # display_board_characters(plyBoard)

print(f"counts: {tempCount}")

print("Summary - random-player vs NN-player")
print(f"Total number of games won by random-player: {tempCount[0]}")
print(f"Total number of games won by NN-player:     {tempCount[1]}")
print(f"Total number of games won drawn:            {tempCount[2]}")



Game is won by O (-1), first move was 1
O | O | O
X | X | O
X | X |  

Game is won by O (-1), first move was 1
O | X | X
O | X |  
O |   |  

Game is won by O (-1), first move was 1
O | X | O
X | X | O
X |   | O

Game is won by O (-1), first move was 1
  |   | O
  | O | X
O | X | X

Game is won by O (-1), first move was 1
O | X | O
X | O |  
O | X | X

Game is a draw, first move was 1
O | O | X
X | X | O
O | X | X

Game is won by O (-1), first move was 1
X |   | X
O | O | O
  |   | X

Game is a draw, first move was 1
X | O | X
O | X | X
O | X | O

Game is won by O (-1), first move was 1
O | X | O
  | O | X
X | X | O

Game is won by O (-1), first move was 1
O | X | X
O | O | O
  | X | X

Game is won by O (-1), first move was 1
O | O | O
  | X |  
X | X |  

Game is won by O (-1), first move was 1
O |   |  
X | O |  
X | X | O

Game is won by O (-1), first move was 1
X |   | O
  | O |  
O | X | X

Game is a draw, first move was 1
X | O | O
O | X | X
X | X | O

Game is won by O (-1), firs

The neural network model was then evaluated by simulating games of the minimax-player vs the neural network player. The minimax-player always goes first to give it the advantage.

As can be seen below, all games result in a draw.

In [35]:
# Evaluate model

player1X = MiniMaxPlayer(playerType = 1, solutionMatrix=solMat)
# player1X = RandomPlayer()
player2O = nn_v1_player(model=model)

countResult = [3]

tempCount = np.zeros(3, dtype=np.int16)

for n in range(50):

    rndBoard = np.zeros(9, dtype=int)   # Set board 

    result = playGameClass(playerX=player1X, playerO=player2O, solMat=solMat, startBoard=rndBoard, startPlayer=1)

    if result > 0:
        tempCount[0] += 1
        # print("Game is won by X (1)")
        # display_board_characters(plyBoard)               
    elif result < 0:
        tempCount[1] += 1
        # print("Game is won by O (-1)")
        # display_board_characters(plyBoard)
    else:
        tempCount[2] += 1
        # print("Game is a draw")
        # display_board_characters(plyBoard)

print(f"counts: {tempCount}")

print("Summary - minimax-player vs NN-player")
print(f"Total number of games won by minimax-player: {tempCount[0]}")
print(f"Total number of games won by NN-player:      {tempCount[1]}")
print(f"Total number of games won drawn:             {tempCount[2]}")


Game is a draw, first move was 1
O | X | O
X | O | X
X | O | X

Game is a draw, first move was 1
O | X | O
X | O | X
X | O | X

Game is a draw, first move was 1
O | X | X
X | O | O
O | X | X

Game is a draw, first move was 1
O | O | X
X | X | O
O | X | X

Game is a draw, first move was 1
O | X | O
X | O | X
X | O | X

Game is a draw, first move was 1
O | X | O
X | O | X
X | O | X

Game is a draw, first move was 1
X | O | O
O | X | X
X | X | O

Game is a draw, first move was 1
O | X | O
X | O | X
X | O | X

Game is a draw, first move was 1
X | O | X
O | O | X
X | X | O

Game is a draw, first move was 1
X | O | X
O | O | X
X | X | O

Game is a draw, first move was 1
X | O | X
O | O | X
X | X | O

Game is a draw, first move was 1
O | X | O
O | X | X
X | O | X

Game is a draw, first move was 1
O | X | O
X | O | X
X | O | X

Game is a draw, first move was 1
X | X | O
O | O | X
X | O | X

Game is a draw, first move was 1
O | X | X
X | X | O
O | O | X

Game is a draw, first move was 1
X | O |